# UHPGC Compressive Strength Prediction - Complete ML Workflow
## Machine Learning Project for Ultra-High-Performance Geopolymer Concrete

This notebook implements a complete machine learning workflow for predicting the compressive strength of Ultra-High-Performance Geopolymer Concrete (UHPGC) using Random Forest, SVR, and XGBoost models.

**Research Paper Reference:**
"Investigation of machine learning models in predicting compressive strength for ultra-high-performance geopolymer concrete: A comparative study"

## Table of Contents
1. [Imports and Setup](#imports)
2. [Data Loading](#data-loading)
3. [Data Preprocessing](#preprocessing)
4. [Exploratory Data Analysis](#eda)
5. [Model Training](#model-training)
6. [Model Evaluation](#evaluation)
7. [Feature Importance Analysis](#feature-importance)
8. [Monte Carlo Simulation](#monte-carlo)
9. [Prediction System](#prediction)
10. [Conclusion](#conclusion)

<a id='imports'></a>
## 1. Imports and Setup

Import all necessary libraries and modules for the project.

In [1]:
# Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

# Import custom modules
from src.data_preprocessing import DataPreprocessor
from src.eda import EDAAnalyzer
from src.model_training import ModelTrainer
from src.evaluation import ModelEvaluator
from src.prediction import PredictionSystem
from src.monte_carlo import MonteCarloSimulation
from src.feature_importance import FeatureImportanceAnalyzer
from src.utils import print_section, save_model, save_scaler

print("All libraries imported successfully!")

All libraries imported successfully!


<a id='data-loading'></a>
## 2. Data Loading

Load the UHPGC dataset from CSV file.

In [2]:
# Define dataset path
# NOTE: Place your dataset.csv file in the data/ folder
data_path = '../data/dataset.csv'

# Check if dataset exists
if os.path.exists(data_path):
    print(f"Dataset found at: {data_path}")
else:
    print(f"WARNING: Dataset not found at {data_path}")
    print("Please place your dataset.csv file in the data/ folder")
    print("For now, we'll create a sample dataset for demonstration")
    
    # Create sample dataset for demonstration
    np.random.seed(42)
    n_samples = 200
    
    sample_data = {
        'Cement_kg_m3': np.random.uniform(500, 800, n_samples),
        'Fly_Ash_kg_m3': np.random.uniform(50, 300, n_samples),
        'Silica_Fume_kg_m3': np.random.uniform(20, 100, n_samples),
        'Metakaolin_kg_m3': np.random.uniform(0, 100, n_samples),
        'GGBS_kg_m3': np.random.uniform(0, 300, n_samples),
        'RHA_kg_m3': np.random.uniform(0, 80, n_samples),
        'POFA_kg_m3': np.random.uniform(0, 80, n_samples),
        'Fine_Sand_kg_m3': np.random.uniform(500, 800, n_samples),
        'Water_kg_m3': np.random.uniform(100, 200, n_samples),
        'Extra_Water_kg_m3': np.random.uniform(10, 50, n_samples),
        'Water_Binder_Ratio': np.random.uniform(0.3, 0.6, n_samples),
        'Na2SiO3_Content_kg_m3': np.random.uniform(50, 150, n_samples),
        'NaOH_Content_kg_m3': np.random.uniform(20, 80, n_samples),
        'KOH_Content_kg_m3': np.random.uniform(5, 30, n_samples),
        'Activator_Molarity_M': np.random.uniform(8, 16, n_samples),
        'Superplasticizer_kg_m3': np.random.uniform(5, 20, n_samples),
        'Polypropylene_Fiber_Content_%': np.random.uniform(0, 2, n_samples),
        'PP_Fiber_kg_m3': np.random.uniform(0, 10, n_samples),
        'Fiber_Length_mm': np.random.uniform(6, 20, n_samples),
        'Curing_Temperature_C': np.random.uniform(25, 90, n_samples),
        'Curing_Duration_days': np.random.uniform(7, 90, n_samples),
        'Compressive_Strength_MPa': np.random.uniform(50, 150, n_samples)  # Target variable
    }
    
    df = pd.DataFrame(sample_data)
    df.to_csv(data_path, index=False)
    print(f"Sample dataset created and saved to: {data_path}")

Dataset found at: ../data/dataset.csv


<a id='preprocessing'></a>
## 3. Data Preprocessing

Clean, preprocess, and prepare the data for machine learning.

In [3]:
# Initialize preprocessor
preprocessor = DataPreprocessor(data_path=data_path, target_column='Compressive_Strength_MPa')

# Run complete preprocessing pipeline
X_train, X_test, y_train, y_test = preprocessor.preprocess_pipeline(
    test_size=0.3,
    random_state=42,
    scaling_method='standard'
)

# Save the scaler for later use
os.makedirs('../models', exist_ok=True)
save_scaler(preprocessor.scaler, '../models/scaler.pkl')

# Store feature names for later use
feature_names = preprocessor.original_feature_names
print(f"\nFeature names: {feature_names}")

                        COMPLETE PREPROCESSING PIPELINE                         
                                  DATA LOADING                                  
Dataset loaded successfully from: ../data/dataset.csv
Dataset shape: (812, 53)
Number of samples: 812
Number of features: 53
                                DATA EXPLORATION                                

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 812 entries, 0 to 811
Data columns (total 53 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Mix_ID                         812 non-null    object 
 1   Cement_kg_m3                   812 non-null    int64  
 2   Fly_Ash_kg_m3                  812 non-null    int64  
 3   Silica_Fume_kg_m3              812 non-null    int64  
 4   Metakaolin_kg_m3               812 non-null    int64  
 5   GGBS_kg_m3                     812 non-null    int64  
 6   RHA_kg_m3           

<a id='eda'></a>
## 4. Exploratory Data Analysis

Perform comprehensive EDA to understand the dataset.

In [4]:
# Initialize EDA analyzer
eda_analyzer = EDAAnalyzer(
    df=preprocessor.df,
    target_column='Compressive_Strength_MPa',
    output_dir='../outputs/graphs'
)

# Generate all EDA visualizations
eda_analyzer.generate_all_plots()

                       GENERATING ALL EDA VISUALIZATIONS                        
                              DATASET INFORMATION                               

Dataset Shape: (812, 53)
Number of Rows: 812
Number of Columns: 53

Column Names:
1. Mix_ID
2. Cement_kg_m3
3. Fly_Ash_kg_m3
4. Silica_Fume_kg_m3
5. Metakaolin_kg_m3
6. GGBS_kg_m3
7. RHA_kg_m3
8. POFA_kg_m3
9. Fine_Sand_kg_m3
10. Water_kg_m3
11. Extra_Water_kg_m3
12. Water_Binder_Ratio
13. Na2SiO3_Content_kg_m3
14. NaOH_Content_kg_m3
15. KOH_Content_kg_m3
16. Activator_Molarity_M
17. Superplasticizer_kg_m3
18. Polypropylene_Fiber_Content_%
19. PP_Fiber_kg_m3
20. Fiber_Length_mm
21. Compressive_Strength_MPa
22. Flexural_Strength_MPa
23. Tensile_Strength_MPa
24. Elastic_Modulus_GPa
25. Interlayer_Bond_Strength_MPa
26. RCPT
27. Water_Absorption_%
28. Sorptivity
29. Gas_Permeability
30. Chloride_Penetration
31. Acid_Attack_Loss_%
32. Sulphate_Resistance_%
33. Printing_Speed_mm_s
34. Layer_Height_mm
35. Nozzle_Diameter_mm
36. Buil

TypeError: could not convert string to float: 'FT-CON'

<a id='model-training'></a>
## 5. Model Training

Train Random Forest, SVR, and XGBoost models with hyperparameter tuning.

In [5]:
# Initialize model trainer
trainer = ModelTrainer(random_state=42)

# Train all models with hyperparameter tuning
# Set tune_hyperparameters=False for faster training (default: True)
models = trainer.train_all_models(
    X_train=X_train,
    y_train=y_train,
    tune_hyperparameters=True
)

# Display training summary
training_summary = trainer.get_training_summary()
print("\nTraining Summary:")
print(training_summary.to_string(index=False))

                              TRAINING ALL MODELS                               
                            RANDOM FOREST REGRESSOR                             
Performing hyperparameter tuning...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Best CV Score: 0.9573
Cross-Validation R² Scores: [0.94053234 0.95446473 0.962216   0.96396702 0.96552707]
Mean CV R² Score: 0.9573 (+/- 0.0184)
Random Forest training completed!
                           SUPPORT VECTOR REGRESSION                            
Performing hyperparameter tuning...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'kernel': 'rbf', 'gamma': 0.01, 'epsilon': 1.0, 'C': 100}
Best CV Score: 0.9542
Cross-Validation R² Scores: [0.94588788 0.93565901 0.96120744 0.96680218 0.96169047]
Mean CV R² Score: 0.9542 (+/- 0.0233)
SVR training completed!
        

In [6]:
# Save all trained models
trainer.save_all_models(output_dir='../models')
print("\nAll models saved successfully!")

                                 SAVING MODELS                                  
Model saved to: ../models\random_forest.pkl
Model saved to: ../models\svr.pkl
Model saved to: ../models\xgboost.pkl
All models saved successfully!

All models saved successfully!


<a id='evaluation'></a>
## 6. Model Evaluation

Evaluate all models using various metrics and generate visualizations.

In [7]:
# Initialize model evaluator
evaluator = ModelEvaluator(output_dir='../outputs')

# Evaluate all models
evaluation_results = evaluator.evaluate_all_models(
    models=models,
    X_test=X_test,
    y_test=y_test
)

# Generate all evaluation plots
evaluator.generate_all_plots()

                             EVALUATING ALL MODELS                              
                            EVALUATING RANDOM FOREST                            
Evaluation Metrics:
--------------------------------------------------------------------------------
R²: 0.9640
MAE: 3.3748
RMSE: 5.1395
MSE: 26.4143
                                 EVALUATING SVR                                 
Evaluation Metrics:
--------------------------------------------------------------------------------
R²: 0.9542
MAE: 4.0011
RMSE: 5.7932
MSE: 33.5606
                               EVALUATING XGBOOST                               
Evaluation Metrics:
--------------------------------------------------------------------------------
R²: 0.9781
MAE: 2.6841
RMSE: 4.0117
MSE: 16.0933
                        GENERATING ALL EVALUATION PLOTS                         
                           ACTUAL VS PREDICTED PLOTS                            
Actual vs Predicted plot for Random Forest saved
Actual vs Predi

In [8]:
# Generate comprehensive evaluation report
report = evaluator.generate_evaluation_report()
print(report)

                          GENERATING EVALUATION REPORT                          
                             MODEL COMPARISON TABLE                             
        Model  R² Score      MAE     RMSE       MSE
      XGBoost  0.978051 2.684086 4.011651 16.093343
Random Forest  0.963974 3.374786 5.139484 26.414300
          SVR  0.954227 4.001118 5.793151 33.560602
DataFrame saved to: ../outputs\metrics\model_comparison_20260525_002734.csv

Comparison table saved to: ../outputs\metrics\model_comparison_20260525_002734.csv
Evaluation report saved to: ../outputs\reports\evaluation_report_20260525_002734.txt
                            MODEL EVALUATION REPORT                             

Generated on: 20260525_002734


MODEL COMPARISON:
--------------------------------------------------------------------------------
        Model  R² Score      MAE     RMSE       MSE
      XGBoost  0.978051 2.684086 4.011651 16.093343
Random Forest  0.963974 3.374786 5.139484 26.414300
          SVR  0

<a id='feature-importance'></a>
## 7. Feature Importance Analysis

Analyze and visualize feature importance for all models.

In [9]:
# Initialize feature importance analyzer
fi_analyzer = FeatureImportanceAnalyzer(
    models=models,
    feature_names=feature_names,
    output_dir='../outputs'
)

# Analyze feature importance for all models
fi_analyzer.analyze_all_models(X_train=X_train, y_train=y_train)

                          FEATURE IMPORTANCE ANALYSIS                           

Analyzing Random Forest...
Top 5 important features for Random Forest:
--------------------------------------------------------------------------------
1. Silica_Fume_kg_m3: 0.1818
2. Superplasticizer_kg_m3: 0.1105
3. Water_kg_m3: 0.1034
4. PP_Fiber_kg_m3: 0.0888
5. Curing_Temperature_C: 0.0861

Analyzing SVR...
Top 5 important features for SVR:
--------------------------------------------------------------------------------
1. Superplasticizer_kg_m3: 0.4277
2. NaOH_Content_kg_m3: 0.3689
3. Na2SiO3_Content_kg_m3: 0.2205
4. Water_kg_m3: 0.0768
5. PP_Fiber_kg_m3: 0.0623

Analyzing XGBoost...
Top 5 important features for XGBoost:
--------------------------------------------------------------------------------
1. Silica_Fume_kg_m3: 0.2354
2. PP_Fiber_kg_m3: 0.1979
3. Curing_Temperature_C: 0.1312
4. Polypropylene_Fiber_Content_%: 0.0829
5. Water_Binder_Ratio: 0.0776


{'Random Forest': Silica_Fume_kg_m3                0.181797
 Superplasticizer_kg_m3           0.110528
 Water_kg_m3                      0.103450
 PP_Fiber_kg_m3                   0.088761
 Curing_Temperature_C             0.086056
 Water_Binder_Ratio               0.084118
 Polypropylene_Fiber_Content_%    0.082527
 Curing_Duration_days             0.081229
 Fine_Sand_kg_m3                  0.037265
 Extra_Water_kg_m3                0.031193
 Cement_kg_m3                     0.026170
 GGBS_kg_m3                       0.020608
 Fiber_Length_mm                  0.018478
 Activator_Molarity_M             0.010905
 Na2SiO3_Content_kg_m3            0.010221
 Fly_Ash_kg_m3                    0.009320
 NaOH_Content_kg_m3               0.006941
 Metakaolin_kg_m3                 0.006210
 RHA_kg_m3                        0.003003
 POFA_kg_m3                       0.001220
 KOH_Content_kg_m3                0.000000
 dtype: float64,
 'SVR': Superplasticizer_kg_m3           0.427740
 NaOH_Content

In [10]:
# Plot feature importance for all models
fi_analyzer.plot_all_importances(top_n=10)

                      GENERATING FEATURE IMPORTANCE PLOTS                       
Feature importance plot for Random Forest saved
Feature importance plot for SVR saved
Feature importance plot for XGBoost saved
                     ALL FEATURE IMPORTANCE PLOTS GENERATED                     


In [11]:
# Plot feature importance comparison across models
fi_analyzer.plot_comparison(top_n=10)

                         FEATURE IMPORTANCE COMPARISON                          
Feature importance comparison plot saved


<Figure size 1400x800 with 0 Axes>

In [12]:
# Generate feature importance report
fi_report = fi_analyzer.generate_summary_report()
print(fi_report)

                      GENERATING FEATURE IMPORTANCE REPORT                      
Feature importance report saved to: ../outputs\reports\feature_importance_report_20260525_002743.txt
                       FEATURE IMPORTANCE ANALYSIS REPORT                       

Generated on: 20260525_002743


Random Forest:
--------------------------------------------------------------------------------

Top 10 Most Important Features:
 1. Silica_Fume_kg_m3             : 0.1818
 2. Superplasticizer_kg_m3        : 0.1105
 3. Water_kg_m3                   : 0.1034
 4. PP_Fiber_kg_m3                : 0.0888
 5. Curing_Temperature_C          : 0.0861
 6. Water_Binder_Ratio            : 0.0841
 7. Polypropylene_Fiber_Content_% : 0.0825
 8. Curing_Duration_days          : 0.0812
 9. Fine_Sand_kg_m3               : 0.0373
10. Extra_Water_kg_m3             : 0.0312

Bottom 5 Least Important Features:
 1. NaOH_Content_kg_m3            : 0.0069
 2. Metakaolin_kg_m3              : 0.0062
 3. RHA_kg_m3          

<a id='monte-carlo'></a>
## 8. Monte Carlo Simulation

Perform 100-run Monte Carlo simulation to analyze model stability.

In [ ]:
# Initialize Monte Carlo simulation
# Note: This may take some time as it runs 100 simulations per model
mc_sim = MonteCarloSimulation(
    X=preprocessor.X,
    y=preprocessor.y,
    models=models,
    n_simulations=100,
    test_size=0.3,
    output_dir='../outputs'
)

# Run all simulations
mc_results = mc_sim.run_all_simulations()

                 RUNNING MONTE CARLO SIMULATIONS FOR ALL MODELS                 
                     MONTE CARLO SIMULATION - RANDOM FOREST                     
Running 100 simulations...
Completed 20/100 simulations


In [ ]:
# Generate all Monte Carlo plots (PDF, CDF, distributions)
mc_sim.plot_all_distributions()

In [ ]:
# Create Monte Carlo summary table
mc_summary = mc_sim.create_summary_table()
print(mc_summary.to_string(index=False))

In [ ]:
# Generate Monte Carlo report
mc_report = mc_sim.generate_report()
print(mc_report)

<a id='prediction'></a>
## 9. Prediction System

Demonstrate the prediction system with sample inputs.

In [ ]:
# Initialize prediction system
predictor = PredictionSystem(
    models_dir='../models',
    scaler_path='../models/scaler.pkl'
)

# Load models
predictor.load_models()

# Set feature names
predictor.set_feature_names(feature_names)

In [ ]:
# Example prediction with sample input
sample_input = {
    'Cement_kg_m3': 780,
    'Fly_Ash_kg_m3': 120,
    'Silica_Fume_kg_m3': 50,
    'Metakaolin_kg_m3': 0,
    'GGBS_kg_m3': 0,
    'RHA_kg_m3': 0,
    'POFA_kg_m3': 0,
    'Fine_Sand_kg_m3': 0,
    'Water_kg_m3': 0,
    'Extra_Water_kg_m3': 0,
    'Water_Binder_Ratio': 0.0,
    'Na2SiO3_Content_kg_m3': 0,
    'NaOH_Content_kg_m3': 0,
    'KOH_Content_kg_m3': 0,
    'Activator_Molarity_M': 0,
    'Superplasticizer_kg_m3': 0,
    'Polypropylene_Fiber_Content_%': 0.0,
    'PP_Fiber_kg_m3': 0,
    'Fiber_Length_mm': 0,
    'Curing_Temperature_C': 25,
    'Curing_Duration_days': 28
}

# Predict using all models
predictions = predictor.predict_with_all_models(sample_input)
print("\nPredictions for sample input:")
for model_name, pred in predictions.items():
    print(f"{model_name}: {pred:.4f} MPa")

<a id='conclusion'></a>
## 10. Conclusion

### Summary of Results

This notebook implemented a complete machine learning workflow for predicting UHPGC compressive strength:

1. **Data Preprocessing**: Cleaned and preprocessed the dataset with scaling
2. **EDA**: Generated comprehensive visualizations to understand the data
3. **Model Training**: Trained RF, SVR, and XGBoost with hyperparameter tuning
4. **Evaluation**: Evaluated models using R², MAE, RMSE, and MSE metrics
5. **Feature Importance**: Identified the most influential features
6. **Monte Carlo**: Performed 100 simulations to analyze model stability
7. **Prediction**: Created a functional prediction system

### Key Findings

- All models were trained and evaluated successfully
- Feature importance analysis shows which materials most affect compressive strength
- Monte Carlo simulation demonstrates model stability across different data splits
- The prediction system is ready for practical use

### Next Steps

- Use the Streamlit app (app.py) for interactive predictions
- Review the generated reports in outputs/reports/
- Examine the visualizations in outputs/graphs/
- Deploy the models for production use

### Files Generated

- Trained models in models/
- Graphs and plots in outputs/graphs/
- Evaluation reports in outputs/reports/
- Metrics in outputs/metrics/

Thank you for using this UHPGC Compressive Strength Prediction System!